# 02. Baseline Model - 스마트 창고 출고 지연 예측

## 전략
1. 기본 전처리 (결측치, 인코딩)
2. LightGBM 베이스라인
3. K-Fold CV로 안정적 평가
4. 제출 파일 생성

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
import os

warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 1. 데이터 로드

In [ ]:
DATA_PATH = '../data/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
layout = pd.read_csv(DATA_PATH + 'layout_info.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train: {train.shape}, Test: {test.shape}')

## 2. 전처리

In [ ]:
# ID 컬럼과 타겟 분리
TARGET = 'avg_delay'

# ID 컬럼 자동 탐지
id_cols = [c for c in train.columns if 'id' in c.lower() or c == 'ID']
print(f'ID 컬럼: {id_cols}')

# 피처 컬럼
feature_cols = [c for c in train.columns if c not in id_cols + [TARGET]]
print(f'피처 수: {len(feature_cols)}')

# 범주형 컬럼 처리
cat_cols = train[feature_cols].select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    print(f'범주형 컬럼: {cat_cols}')
    for col in cat_cols:
        train[col] = train[col].astype('category').cat.codes
        test[col] = test[col].astype('category').cat.codes

X = train[feature_cols].values
y = train[TARGET].values
X_test = test[feature_cols].values

print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')

## 3. LightGBM Baseline (5-Fold CV)

In [ ]:
N_FOLDS = 5
SEED = 42

params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'n_jobs': -1,
    'seed': SEED,
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f'\n--- Fold {fold + 1}/{N_FOLDS} ---')
    
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    train_data = lgb.Dataset(X_train, y_train)
    val_data = lgb.Dataset(X_val, y_val, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=2000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(200)
        ]
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / N_FOLDS
    
    fold_mae = mean_absolute_error(y_val, oof_preds[val_idx])
    fold_scores.append(fold_mae)
    print(f'Fold {fold + 1} MAE: {fold_mae:.6f}')

overall_mae = mean_absolute_error(y, oof_preds)
print(f'\n=== Overall OOF MAE: {overall_mae:.6f} ===')
print(f'=== Fold MAEs: {[f"{s:.6f}" for s in fold_scores]} ===')
print(f'=== Std: {np.std(fold_scores):.6f} ===')

## 4. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = model.feature_importance(importance_type='gain')
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 10))
plt.barh(feat_imp['feature'].head(30), feat_imp['importance'].head(30), color='steelblue')
plt.title('Top 30 Feature Importance (Gain)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. 제출 파일 생성

In [ ]:
submission[TARGET] = test_preds

# 음수 예측값 클리핑 (지연 시간은 0 이상)
submission[TARGET] = submission[TARGET].clip(lower=0)

SUBMISSION_PATH = '../submissions/'
os.makedirs(SUBMISSION_PATH, exist_ok=True)

filename = f'baseline_lgbm_mae{overall_mae:.4f}.csv'
submission.to_csv(SUBMISSION_PATH + filename, index=False)

print(f'제출 파일 저장: {filename}')
print(f'\n제출 파일 미리보기:')
submission.head()

## 6. 다음 단계

### 성능 개선 아이디어
- [ ] layout_info.csv 조인하여 피처 추가
- [ ] 타임슬롯 기반 시계열 피처 엔지니어링
- [ ] 시나리오 내 lag/rolling 피처
- [ ] XGBoost, CatBoost 앙상블
- [ ] 하이퍼파라미터 튜닝 (Optuna)
- [ ] 타겟 로그 변환 실험
- [ ] 이상치 처리